# Kaggle Submission Generator
**F20AA CW2 — Google Maps Review Challenge**

This notebook generates a Kaggle-ready `submission.csv` using:
- The saved TF-IDF vectorizer from Task 4 (`tfidf_vectorizer.pkl`)
- Logistic Regression retrained with best hyperparameters (C=1.0, L1)
- The same preprocessing pipeline from Task 2

> **Before running:** Download `test.csv` from the Kaggle competition Data tab and place it in the `data/` folder.

## 1. Imports

In [1]:
import pandas as pd
import numpy as np
import pickle
import re
import string
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from sklearn.linear_model import LogisticRegression

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

print('Imports done.')

Imports done.


## 2. Preprocessing Pipeline (same as Task 2)

In [2]:
lemmatizer = WordNetLemmatizer()

STOPWORDS = set(stopwords.words('english'))
KEEP_NEGATIONS = {'not', 'never', 'no', 'nor', 'none'}
STOPWORDS -= KEEP_NEGATIONS  # keep negation words

def remove_emojis(text):
    emoji_pattern = re.compile(
        '['
        u'\U0001F600-\U0001F64F'
        u'\U0001F300-\U0001F5FF'
        u'\U0001F680-\U0001F9FF'
        u'\U00002702-\U000027B0'
        u'\U000024C2-\U0001F251'
        ']+', flags=re.UNICODE)
    return emoji_pattern.sub('', text)

def handle_negations(text):
    text = re.sub(r"n't", ' not', text)
    text = re.sub(r"won't", 'will not', text)
    text = re.sub(r"can't", 'cannot', text)
    return text

def preprocess(text):
    if not isinstance(text, str):
        return ''
    text = remove_emojis(text)                          # remove emojis
    text = re.sub(r'<[^>]+>', ' ', text)                # strip HTML
    text = re.sub(r'http\S+|www\.\S+', '', text)        # remove URLs
    text = text.lower()                                  # lowercase
    text = handle_negations(text)                        # negation handling
    text = text.translate(str.maketrans('', '', string.punctuation))  # remove punctuation
    tokens = word_tokenize(text)                         # tokenize
    tokens = [t for t in tokens if t not in STOPWORDS]  # remove stopwords
    tokens = [lemmatizer.lemmatize(t) for t in tokens]  # lemmatize
    return ' '.join(tokens)

# Quick sanity check
sample = "I can't believe how TERRIBLE this place is! Never going back."
print('Input: ', sample)
print('Output:', preprocess(sample))

Input:  I can't believe how TERRIBLE this place is! Never going back.
Output: ca not believe terrible place never going back


## 3. Load Saved TF-IDF Vectorizer

In [3]:
with open('tfidf_vectorizer.pkl', 'rb') as f:
    tfidf = pickle.load(f)

print(f'Vectorizer loaded. Vocabulary size: {len(tfidf.vocabulary_):,}')
print(f'N-gram range: {tfidf.ngram_range}')
print(f'Max features: {tfidf.max_features}')

Vectorizer loaded. Vocabulary size: 10,000
N-gram range: (1, 2)
Max features: 10000


## 4. Load & Preprocess Training Data, Train LR Model

In [4]:
print('Loading training data...')
train = pd.read_csv('data/train_english.csv')  # English-only filtered data from Task 2
print(f'Train shape: {train.shape}')
train.head(2)

Loading training data...
Train shape: (271897, 3)


,text,rating,text_length
0,This place is TERRIBLE; the people in charge a...,2,551
1,Terrible Service! And they are saying that I n...,1,258


In [5]:
print('Preprocessing training texts...')
train['cleaned'] = train['text'].apply(preprocess)
print('Done.')

Preprocessing training texts...
Done.


In [6]:
print('Transforming training data with saved TF-IDF vectorizer...')
X_train = tfidf.transform(train['cleaned'])
y_train = train['rating']
print(f'Feature matrix shape: {X_train.shape}')

Transforming training data with saved TF-IDF vectorizer...
Feature matrix shape: (271897, 10000)


In [7]:
print('Training Logistic Regression (best params from Task 4: C=1.0, L1)...')
lr = LogisticRegression(
    C=1.0,
    penalty='l1',
    solver='liblinear',
    class_weight=None,
    max_iter=1000,
    random_state=42
)
lr.fit(X_train, y_train)
print('Model trained.')

Training Logistic Regression (best params from Task 4: C=1.0, L1)...


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1296: FutureWarning: Using the 'liblinear' solver for multiclass classification is deprecated. An error will be raised in 1.8. Either use another solver which supports the multinomial loss or wrap the estimator in a OneVsRestClassifier to keep applying a one-versus-rest scheme.
  warnings.warn(


Model trained.


## 5. Load Kaggle Test Data & Generate Predictions

> Download `test.csv` from the Kaggle competition **Data** tab and place it in `data/kaggle_test.csv`

In [8]:
print('Loading Kaggle test data...')
test = pd.read_csv('data/kaggle_test.csv')
print(f'Test shape: {test.shape}')
print(f'Columns: {test.columns.tolist()}')
test.head()

Loading Kaggle test data...
Test shape: (89100, 2)
Columns: ['Id', 'text']


,Id,text
0,1,Quite easy to rent a car bur it is not easy to...
1,2,Nice voleyball court close to restaurants and ...
2,3,"Very nice built homes, the future locations ar..."
3,4,This dental clinic appears to be friendly with...
4,5,We came in to discuss tattoos. Only person th...


In [9]:
# Normalise column names (handle capitalisation differences)
test.columns = [c.strip().lower() for c in test.columns]

# Rename to standard names if needed
if 'text' not in test.columns:
    # Try common alternatives
    for col in ['review', 'review_text', 'Text']:
        if col.lower() in test.columns:
            test = test.rename(columns={col.lower(): 'text'})
            break

print(f'Columns after normalisation: {test.columns.tolist()}')
print(f'Total test rows: {len(test):,}')

Columns after normalisation: ['id', 'text']
Total test rows: 89,100


In [10]:
print('Preprocessing test texts...')
test['cleaned'] = test['text'].apply(preprocess)
print('Done.')

Preprocessing test texts...
Done.


In [11]:
print('Transforming test data...')
X_test = tfidf.transform(test['cleaned'])
print(f'Test feature matrix shape: {X_test.shape}')

Transforming test data...
Test feature matrix shape: (89100, 10000)


In [12]:
print('Generating predictions...')
predictions = lr.predict(X_test)

print(f'Predictions shape: {predictions.shape}')
print(f'Unique predicted ratings: {sorted(set(predictions))}')
print('\nPrediction distribution:')
pred_series = pd.Series(predictions)
print(pred_series.value_counts().sort_index())

Generating predictions...
Predictions shape: (89100,)
Unique predicted ratings: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]

Prediction distribution:
1    21854
2     3141
3     6352
4    52052
5     5701
Name: count, dtype: int64


## 6. Save Submission CSV

In [13]:
# Build submission dataframe
# The Id column may be named 'id' or 'Id'
id_col = 'id' if 'id' in test.columns else test.columns[0]

submission = pd.DataFrame({
    'Id': test[id_col].values,
    'Rating': predictions
})

print(f'Submission shape: {submission.shape}')
submission.head(10)

Submission shape: (89100, 2)


,Id,Rating
0,1,4
1,2,4
2,3,4
3,4,5
4,5,5
5,6,4
6,7,4
7,8,4
8,9,1
9,10,4


In [14]:
# Validate before saving
assert submission.shape[1] == 2, 'Must have exactly 2 columns'
assert list(submission.columns) == ['Id', 'Rating'], f'Wrong column names: {submission.columns.tolist()}'
assert submission['Rating'].between(1, 5).all(), 'Ratings must be between 1 and 5'
assert submission['Id'].is_unique, 'IDs must be unique'
assert not submission.isnull().any().any(), 'No nulls allowed'

print(f'All checks passed!')
print(f'Rows: {len(submission):,}')
print(f'Columns: {submission.columns.tolist()}')
print(f'Rating range: {submission["Rating"].min()} - {submission["Rating"].max()}')

All checks passed!
Rows: 89,100
Columns: ['Id', 'Rating']
Rating range: 1 - 5


In [15]:
output_path = 'submission_LR.csv'
submission.to_csv(output_path, index=False)
print(f'Saved to: {output_path}')

# Verify saved file
verify = pd.read_csv(output_path)
print(f'Verified rows: {len(verify):,}')
print(verify.head())

Saved to: submission_LR.csv
Verified rows: 89,100
   Id  Rating
0   1       4
1   2       4
2   3       4
3   4       5
4   5       5
